In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [6]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\my Files\RiskPredANDMarketEvaluation")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "market_data.csv"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 10
THRESHOLD = 0.05

ACCEPTED_TICKERS = ["NVDA", "AMD", "INTC", "TSM"]

print("Project root exists:", PROJECT_ROOT.exists())
print("Input dataset exists:", DATA_PATH.exists())
print("Output directory:", OUTPUT_DIR)

Project root exists: True
Input dataset exists: True
Output directory: C:\my Files\RiskPredANDMarketEvaluation\experiments\outputs


In [7]:

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"])

required_columns = [
    "date", "ticker", "open", "high", "low",
    "close", "adj_close", "volume",
]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

df = df[df["ticker"].isin(ACCEPTED_TICKERS)].copy()
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Tickers:", sorted(df["ticker"].unique()))
print("Date range:", df["date"].min(), "to", df["date"].max())

display(df.head())


Dataset shape: (5016, 10)
Tickers: ['AMD', 'INTC', 'NVDA', 'TSM']
Date range: 2021-10-15 00:00:00 to 2026-10-14 00:00:00


,date,ticker,open,high,low,close,adj_close,volume,dividends,stock_splits
0,2021-10-15,AMD,112.510002,112.839996,111.110001,112.120003,112.120003,34203000,0.0,0.0
1,2021-10-18,AMD,111.900002,116.879997,111.370003,116.430000,116.430000,47537800,0.0,0.0
2,2021-10-19,AMD,116.160004,117.169998,114.470001,116.330002,116.330002,39593900,0.0,0.0
3,2021-10-20,AMD,116.290001,118.300003,115.419998,116.389999,116.389999,34244800,0.0,0.0
4,2021-10-21,AMD,116.500000,119.680000,116.260002,119.330002,119.330002,34794900,0.0,0.0


In [8]:

print("Duplicate stock-date combinations:",
      df.duplicated(["date", "ticker"]).sum())

print("Missing adjusted closes:", df["adj_close"].isna().sum())

print("Non-positive adjusted closes:",
      (df["adj_close"].dropna() <= 0).sum())

print("Negative volume rows:",
      (df["volume"].dropna() < 0).sum())

print("\nRows per stock:")
print(df.groupby("ticker").size())


Duplicate stock-date combinations: 0
Missing adjusted closes: 0
Non-positive adjusted closes: 0
Negative volume rows: 0

Rows per stock:
ticker
AMD     1254
INTC    1254
NVDA    1254
TSM     1254
dtype: int64


In [10]:

def engineer_features(group):
    group = group.sort_values("date").copy()

    price = group["adj_close"]
    daily_return = price.pct_change(fill_method=None)

    # 1. Price returns
    group["return_1d"] = daily_return
    group["return_5d"] = price.pct_change(5, fill_method=None)
    group["return_10d"] = price.pct_change(10, fill_method=None)

    # 2. Rolling volatility
    group["volatility_5d"] = daily_return.rolling(5).std()
    group["volatility_20d"] = daily_return.rolling(20).std()

    # 3. Drawdown from the running historical peak
    running_peak = price.cummax()
    group["drawdown"] = price / running_peak - 1

    # 4. Trading-volume features
    previous_volume = group["volume"].shift(1)

    group["volume_change_1d"] = (
        group["volume"] / previous_volume - 1
    ).where(previous_volume > 0)

    previous_20d_volume = group["volume"].shift(1).rolling(20).mean()

    group["volume_ratio_20d"] = (
        group["volume"] / previous_20d_volume
    ).where(previous_20d_volume > 0)

    # 5. Future target
    # Is any of the next 10 closing prices at least 5% below today's price?
    future_prices = pd.concat(
        [
            price.shift(-step).rename(f"day_{step}")
            for step in range(1, HORIZON + 1)
        ],
        axis=1,
    )

    complete_window = future_prices.notna().all(axis=1)
    future_min = future_prices.min(axis=1)

    group["stress_event_5pct_10d"] = np.nan

    group.loc[complete_window, "stress_event_5pct_10d"] = (
        future_min.loc[complete_window]
        <= price.loc[complete_window] * (1 - THRESHOLD)
    ).astype(int)

    return group


In [11]:

df_features = pd.concat(
    [
        engineer_features(group)
        for _, group in df.groupby("ticker", sort=False)
    ],
    ignore_index=True,
)

df_features = df_features.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

print("Feature dataset shape:", df_features.shape)

display(
    df_features[
        [
            "date", "ticker", "adj_close",
            "return_1d", "return_5d",
            "return_10d", "volatility_20d",
            "drawdown", "volume_ratio_20d",
            "stress_event_5pct_10d",
        ]
    ].head(15)
)


Feature dataset shape: (5016, 19)


,date,ticker,adj_close,return_1d,return_5d,return_10d,volatility_20d,drawdown,volume_ratio_20d,stress_event_5pct_10d
0,2021-10-15,AMD,112.120003,NaN,NaN,NaN,NaN,0.000000,NaN,0.0
1,2021-10-18,AMD,116.430000,0.038441,NaN,NaN,NaN,0.000000,NaN,0.0
2,2021-10-19,AMD,116.330002,-0.000859,NaN,NaN,NaN,-0.000859,NaN,0.0
3,2021-10-20,AMD,116.389999,0.000516,NaN,NaN,NaN,-0.000344,NaN,0.0
4,2021-10-21,AMD,119.330002,0.025260,NaN,NaN,NaN,0.000000,NaN,0.0
5,2021-10-22,AMD,119.820000,0.004106,0.068676,NaN,NaN,0.000000,NaN,0.0
6,2021-10-25,AMD,122.360001,0.021198,0.050932,NaN,NaN,0.000000,NaN,0.0
7,2021-10-26,AMD,122.930000,0.004658,0.056735,NaN,NaN,0.000000,NaN,0.0
8,2021-10-27,AMD,122.279999,-0.005288,0.050606,NaN,NaN,-0.005288,NaN,0.0
9,2021-10-28,AMD,121.160004,-0.009159,0.015336,NaN,NaN,-0.014398,NaN,0.0


In [12]:

feature_output_path = OUTPUT_DIR / "market_data_with_features.csv"

df_features.to_csv(feature_output_path, index=False)

print("Saved:", feature_output_path)
print("File exists:", feature_output_path.exists())


Saved: C:\my Files\RiskPredANDMarketEvaluation\experiments\outputs\market_data_with_features.csv
File exists: True


In [14]:

feature_columns = [
    "return_1d",
    "return_5d",
    "return_10d",
    "volatility_5d",
    "volatility_20d",
    "drawdown",
    "volume_change_1d",
    "volume_ratio_20d",
]

target_column = "stress_event_5pct_10d"

print("Duplicate stock-date rows:",
      df_features.duplicated(["date", "ticker"]).sum())

print("\nInfinite values:")
print(
    pd.Series({
        col: np.isinf(df_features[col]).sum()
        for col in feature_columns
    })
)

print("\nMissing values:")
print(df_features[feature_columns + [target_column]].isna().sum())

valid_targets = df_features[target_column].dropna()

print("\nOnly binary target labels:",
      valid_targets.isin([0, 1]).all())

print("\nTarget distribution:")
print(df_features[target_column].value_counts(dropna=False))


Duplicate stock-date rows: 0

Infinite values:
return_1d           0
return_5d           0
return_10d          0
volatility_5d       0
volatility_20d      0
drawdown            0
volume_change_1d    0
volume_ratio_20d    0
dtype: int64

Missing values:
return_1d                 4
return_5d                20
return_10d               40
volatility_5d            20
volatility_20d           80
drawdown                  0
volume_change_1d          4
volume_ratio_20d         80
stress_event_5pct_10d    40
dtype: int64

Only binary target labels: True

Target distribution:
stress_event_5pct_10d
0.0    2984
1.0    1992
NaN      40
Name: count, dtype: int64


In [15]:

check = df_features.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

reconstructed_labels = pd.Series(
    np.nan, index=check.index, dtype=float
)

for _, indices in check.groupby("ticker", sort=False).groups.items():
    indices = list(indices)
    prices = check.loc[indices, "adj_close"].reset_index(drop=True)

    future_prices = pd.concat(
        [prices.shift(-step) for step in range(1, HORIZON + 1)],
        axis=1,
    )

    complete = future_prices.notna().all(axis=1)
    future_min = future_prices.min(axis=1)

    labels = pd.Series(np.nan, index=prices.index, dtype=float)

    labels.loc[complete] = (
        future_min.loc[complete]
        <= prices.loc[complete] * (1 - THRESHOLD)
    ).astype(int)

    reconstructed_labels.loc[indices] = labels.to_numpy()

actual = check[target_column].reset_index(drop=True)
expected = reconstructed_labels.reset_index(drop=True)

comparable = actual.notna() & expected.notna()

print("Comparable target rows:", comparable.sum())
print("Target mismatches:",
      (actual[comparable] != expected[comparable]).sum())

print("Target missingness matches:",
      actual.isna().equals(expected.isna()))


Comparable target rows: 4976
Target mismatches: 0
Target missingness matches: True


In [16]:

check = df_features.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

reconstructed_labels = pd.Series(
    np.nan, index=check.index, dtype=float
)

for _, indices in check.groupby("ticker", sort=False).groups.items():
    indices = list(indices)
    prices = check.loc[indices, "adj_close"].reset_index(drop=True)

    future_prices = pd.concat(
        [prices.shift(-step) for step in range(1, HORIZON + 1)],
        axis=1,
    )

    complete = future_prices.notna().all(axis=1)
    future_min = future_prices.min(axis=1)

    labels = pd.Series(np.nan, index=prices.index, dtype=float)

    labels.loc[complete] = (
        future_min.loc[complete]
        <= prices.loc[complete] * (1 - THRESHOLD)
    ).astype(int)

    reconstructed_labels.loc[indices] = labels.to_numpy()

actual = check[target_column].reset_index(drop=True)
expected = reconstructed_labels.reset_index(drop=True)

comparable = actual.notna() & expected.notna()

print("Comparable target rows:", comparable.sum())
print("Target mismatches:",
      (actual[comparable] != expected[comparable]).sum())

print("Target missingness matches:",
      actual.isna().equals(expected.isna()))


Comparable target rows: 4976
Target mismatches: 0
Target missingness matches: True


In [17]:

sample_ticker = "AMD"

sample = (
    df_features[df_features["ticker"] == sample_ticker]
    .sort_values("date")
    .reset_index(drop=True)
)

cutoff_position = len(sample) // 2
cutoff_date = sample.loc[cutoff_position, "date"]

past_only = sample.loc[sample["date"] <= cutoff_date].copy()
recalculated = engineer_features(past_only)

original_row = sample.loc[
    sample["date"] == cutoff_date, feature_columns
].iloc[0]

recalculated_row = recalculated.loc[
    recalculated["date"] == cutoff_date, feature_columns
].iloc[0]

print(
    "Features unchanged:",
    np.allclose(
        original_row.to_numpy(dtype=float),
        recalculated_row.to_numpy(dtype=float),
        equal_nan=True,
        atol=1e-10,
    )
)


Features unchanged: True


In [18]:

model_columns = [
    "date",
    "ticker",
    *feature_columns,
    target_column,
]

model_data = df_features[model_columns].copy()

model_data[feature_columns] = (
    model_data[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
)

rows_before = len(model_data)

model_data = model_data.dropna(
    subset=feature_columns + [target_column]
).copy()

model_data[target_column] = model_data[target_column].astype("int8")

print("Rows before filtering:", rows_before)
print("Rows available for modeling:", len(model_data))

print("\nRows per ticker:")
print(model_data.groupby("ticker").size())

print("\nTarget balance by ticker:")
print(
    model_data.groupby("ticker")[target_column]
    .agg(["count", "sum", "mean"])
    .assign(
        event_rate_pct=lambda x: x["mean"] * 100
    )
    .round(2)
)

candidate_path = OUTPUT_DIR / "model_ready_candidates.csv"
model_data.to_csv(candidate_path, index=False)

print("\nSaved:", candidate_path)
print("File exists:", candidate_path.exists())


Rows before filtering: 5016
Rows available for modeling: 4896

Rows per ticker:
ticker
AMD     1224
INTC    1224
NVDA    1224
TSM     1224
dtype: int64

Target balance by ticker:
        count  sum  mean  event_rate_pct
ticker                                  
AMD      1224  574  0.47           46.90
INTC     1224  550  0.45           44.93
NVDA     1224  491  0.40           40.11
TSM      1224  369  0.30           30.15

Saved: C:\my Files\RiskPredANDMarketEvaluation\experiments\outputs\model_ready_candidates.csv
File exists: True
